In [9]:
# !pip install langchain-chroma

In [10]:
# !pip install langchain-experimental

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
import shutil
import os

In [12]:
import json

with open("college_policies_rag.json", encoding="utf-8") as f:
    policies = json.load(f)

print(type(policies))
print(len(policies))
data = policies

<class 'dict'>
2


In [13]:
ground_truth = [

# BAD-POL-001
{
    "question":"Who approves proposed curriculum changes?",
    "ground_truth_answer":"Academic Council",
    "policy_id":"BAD-POL-001",
    "answer_type":"Role"
},
{
    "question":"How often must academic programs undergo internal review?",
    "ground_truth_answer":"Every five years",
    "policy_id":"BAD-POL-001",
    "answer_type":"Time"
},
{
    "question":"When is the Academic Program Review Policy applied?",
    "ground_truth_answer":"When proposing a new program, revising curriculum, responding to accreditation feedback, or discontinuing a program.",
    "policy_id":"BAD-POL-001",
    "answer_type":"Decision Context"
},
{
    "question":"Who updates the program catalog after approval?",
    "ground_truth_answer":"Registrar Office",
    "policy_id":"BAD-POL-001",
    "answer_type":"Role"
},
{
    "question":"What happens if a proposal is deferred?",
    "ground_truth_answer":"Additional stakeholder consultation is required.",
    "policy_id":"BAD-POL-001",
    "answer_type":"Outcome"
},

# BAD-POL-002
{
    "question":"Who investigates academic misconduct?",
    "ground_truth_answer":"Department Discipline Committee",
    "policy_id":"BAD-POL-002",
    "answer_type":"Role"
},
{
    "question":"How long must investigations be completed within?",
    "ground_truth_answer":"21 working days",
    "policy_id":"BAD-POL-002",
    "answer_type":"Time"
},
{
    "question":"What is the penalty for a first plagiarism offence?",
    "ground_truth_answer":"Grade penalty (zero on the assignment)",
    "policy_id":"BAD-POL-002",
    "answer_type":"Outcome"
},
{
    "question":"Who handles appeals for severe academic misconduct?",
    "ground_truth_answer":"Academic Council",
    "policy_id":"BAD-POL-002",
    "answer_type":"Role"
},
{
    "question":"What tool must be used for major assessments?",
    "ground_truth_answer":"Plagiarism detection tool",
    "policy_id":"BAD-POL-002",
    "answer_type":"Constraint"
},

# BAD-POL-003
{
    "question":"Who signs institutional MOUs?",
    "ground_truth_answer":"College Principal",
    "policy_id":"BAD-POL-003",
    "answer_type":"Role"
},
{
    "question":"How often must MOUs be renewed?",
    "ground_truth_answer":"Every three years",
    "policy_id":"BAD-POL-003",
    "answer_type":"Time"
},
{
    "question":"Who validates legal compliance before signing an MOU?",
    "ground_truth_answer":"Legal/Compliance Officer",
    "policy_id":"BAD-POL-003",
    "answer_type":"Role"
},
{
    "question":"What happens if an MOU expires without renewal?",
    "ground_truth_answer":"Partnership activities are suspended until renewal.",
    "policy_id":"BAD-POL-003",
    "answer_type":"Outcome"
},
{
    "question":"When is the MOU Governance Policy used?",
    "ground_truth_answer":"When formalizing collaborations with external organizations.",
    "policy_id":"BAD-POL-003",
    "answer_type":"Decision Context"
},

# BAD-POL-004
{
    "question":"Who approves internship placement organizations?",
    "ground_truth_answer":"Internship Coordinator",
    "policy_id":"BAD-POL-004",
    "answer_type":"Role"
},
{
    "question":"What must students submit every week?",
    "ground_truth_answer":"Weekly internship logs",
    "policy_id":"BAD-POL-004",
    "answer_type":"Requirement"
},
{
    "question":"What happens if a student fails the internship?",
    "ground_truth_answer":"The student must repeat the internship placement.",
    "policy_id":"BAD-POL-004",
    "answer_type":"Outcome"
},
{
    "question":"Who maintains internship MOU records?",
    "ground_truth_answer":"HR Department",
    "policy_id":"BAD-POL-004",
    "answer_type":"Role"
},
{
    "question":"Can internships be completed at unapproved organizations?",
    "ground_truth_answer":"No. Host organizations must be pre-approved.",
    "policy_id":"BAD-POL-004",
    "answer_type":"Constraint"
}
]

In [14]:
ground_truth.extend([

{
"question":"Who monitors network usage for policy violations?",
"ground_truth_answer":"Network Administrator",
"policy_id":"ITD-POL-001",
"answer_type":"Role"
},
{
"question":"How often must users re-sign the Acceptable Use Policy?",
"ground_truth_answer":"Annually",
"policy_id":"ITD-POL-001",
"answer_type":"Time"
},
{
"question":"What happens after confirmed AUP violations?",
"ground_truth_answer":"IT access is suspended pending investigation.",
"policy_id":"ITD-POL-001",
"answer_type":"Outcome"
},
{
"question":"Which role oversees data classification compliance?",
"ground_truth_answer":"Data Protection Officer (DPO)",
"policy_id":"ITD-POL-002",
"answer_type":"Role"
},
{
"question":"How long must student records be retained?",
"ground_truth_answer":"10 years after graduation",
"policy_id":"ITD-POL-002",
"answer_type":"Time"
},
{
"question":"Can restricted data be transmitted through unencrypted channels?",
"ground_truth_answer":"No",
"policy_id":"ITD-POL-002",
"answer_type":"Constraint"
},
{
"question":"Who leads cybersecurity incident containment?",
"ground_truth_answer":"Incident Response Team (IRT)",
"policy_id":"ITD-POL-003",
"answer_type":"Role"
},
{
"question":"Within how many hours must Level 1 incidents be contained?",
"ground_truth_answer":"4 hours",
"policy_id":"ITD-POL-003",
"answer_type":"Time"
},
{
"question":"Who approves public communication during cyber incidents?",
"ground_truth_answer":"College Administration",
"policy_id":"ITD-POL-003",
"answer_type":"Role"
},
{
"question":"Who maintains the software asset register?",
"ground_truth_answer":"IT Head",
"policy_id":"ITD-POL-004",
"answer_type":"Role"
},
{
"question":"Can software be installed without IT approval?",
"ground_truth_answer":"No",
"policy_id":"ITD-POL-004",
"answer_type":"Constraint"
},
{
"question":"Who executes daily backups?",
"ground_truth_answer":"Backup Administrator",
"policy_id":"ITD-POL-005",
"answer_type":"Role"
},
{
"question":"What is the recovery time objective for critical systems?",
"ground_truth_answer":"4 hours",
"policy_id":"ITD-POL-005",
"answer_type":"Time"
},
{
"question":"Where must backups be stored?",
"ground_truth_answer":"Off-site or in a geographically separate cloud region",
"policy_id":"ITD-POL-005",
"answer_type":"Constraint"
}
])

In [15]:
ground_truth.extend([

{
"question":"Who consolidates departmental budget submissions?",
"ground_truth_answer":"Finance Officer",
"policy_id":"FIN-POL-001",
"answer_type":"Role"
},
{
"question":"When are budget requests due?",
"ground_truth_answer":"Chaitra 15",
"policy_id":"FIN-POL-001",
"answer_type":"Time"
},
{
"question":"Who approves the institutional budget?",
"ground_truth_answer":"College Principal",
"policy_id":"FIN-POL-001",
"answer_type":"Role"
},
{
"question":"How many quotations are required for purchases above NPR 500,000?",
"ground_truth_answer":"Three competitive quotations",
"policy_id":"FIN-POL-002",
"answer_type":"Constraint"
},
{
"question":"When is a public tender required?",
"ground_truth_answer":"For purchases above NPR 5,000,000",
"policy_id":"FIN-POL-002",
"answer_type":"Constraint"
},
{
"question":"Who reviews high-value procurements?",
"ground_truth_answer":"Internal Auditor",
"policy_id":"FIN-POL-002",
"answer_type":"Role"
},
{
"question":"How many days qualify for a full tuition refund?",
"ground_truth_answer":"Withdrawal within 7 days",
"policy_id":"FIN-POL-003",
"answer_type":"Time"
},
{
"question":"How long does a refund take after approval?",
"ground_truth_answer":"15 working days",
"policy_id":"FIN-POL-003",
"answer_type":"Time"
},
{
"question":"Who confirms enrollment status for refunds?",
"ground_truth_answer":"Registrar",
"policy_id":"FIN-POL-003",
"answer_type":"Role"
},
{
"question":"Within how many days must management respond to audit findings?",
"ground_truth_answer":"21 days",
"policy_id":"FIN-POL-004",
"answer_type":"Time"
}
])

In [16]:
embedding_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE": "BAAI/bge-small-en-v1.5",
    "Nomic": "nomic-ai/nomic-embed-text-v1.5"
}

In [17]:
def benchmark_embeddings(documents,
                         strategy_name,
                         persist_directory="vectorstores"):
    """
    documents:
        Output of your chunking function.

    strategy_name:
        Example:
            WholePolicy
            FixedChunk
            SectionChunk
            SemanticChunk
    """

    results = {}

    docs = [
        Document(
            page_content=d["text"],
            metadata=d["metadata"]
        )
        for d in documents
    ]

    for model_name, model in embedding_models.items():

        print("=" * 60)
        print(f"Chunking : {strategy_name}")
        print(f"Embedding: {model_name}")
        print("=" * 60)

        embedding = HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs={
                "trust_remote_code": True
            }
        )

        db_path = os.path.join(
            persist_directory,
            f"{strategy_name}_{model_name}"
        )

        if os.path.exists(db_path):
            shutil.rmtree(db_path)

        vectordb = Chroma.from_documents(
            docs,
            embedding,
            persist_directory=db_path
        )

        retriever = vectordb.as_retriever(
            search_kwargs={
                "k": 3
            }
        )

        results[model_name] = retriever

        print("Done.\n")

    return results

In [18]:
def whole_policy_chunking(data):

    documents = []

    for department in data["departments"]:
        for policy in department["policies"]:

            documents.append({
                "id": policy["policy_id"],
                "text": policy["rag_chunk"],
                "metadata": {
                    "policy_id": policy["policy_id"],      # <-- ADD THIS
                    "department": policy["department_name"],
                    "category": policy["category"],
                    "title": policy["title"]
                }
            })

    return documents

In [19]:
documents = whole_policy_chunking(data)

whole_policy_retrievers = benchmark_embeddings(
    documents,
    "WholePolicy"
)

Chunking : WholePolicy
Embedding: MiniLM
Done.

Chunking : WholePolicy
Embedding: BGE
Done.

Chunking : WholePolicy
Embedding: Nomic


<All keys matched successfully>


Done.



In [20]:
# question = "Who approves curriculum changes?"

# for model_name, retriever in whole_policy_retrievers.items():

#     print(model_name)

#     docs = retriever.invoke(question)

#     print(docs[0].page_content)

In [21]:
import pandas as pd

results = []

for model_name, retriever in whole_policy_retrievers.items():

    print(f"\nTesting {model_name}")

    for sample in ground_truth:

        question = sample["question"]
        expected = sample["policy_id"]

        docs = retriever.invoke(question)

        retrieved_ids = [
            doc.metadata.get("policy_id", "Unknown")
            for doc in docs
        ]

        correct = expected in retrieved_ids

        results.append({
            "Embedding": model_name,
            "Question": question,
            "Expected": expected,
            "Retrieved Top-1": retrieved_ids[0] if retrieved_ids else None,
            "Retrieved IDs": ", ".join(retrieved_ids),
            "Correct": correct
        })

        print("-"*80)
        print("Question :", question)
        print("Expected :", expected)
        print("Retrieved:", retrieved_ids)
        print("PASS" if correct else "FAIL")

        for i, doc in enumerate(docs, 1):

            print(f"\nChunk {i}")
            print("Policy:", doc.metadata.get("policy_id"))
            print(doc.page_content[:300], "...")


Testing MiniLM
--------------------------------------------------------------------------------
Question : Who approves proposed curriculum changes?
Expected : BAD-POL-001
Retrieved: ['BAD-POL-001', 'MGT-POL-001', 'QAA-POL-003']
PASS

Chunk 1
Policy: BAD-POL-001
Policy BAD-POL-001: Academic Program Review and Approval Policy. Department: Business Administration Department (Operational Academic Unit). Category: Curriculum Governance. This policy defines the process for initiating, reviewing, modifying, or discontinuing academic programs. Programs must underg ...

Chunk 2
Policy: MGT-POL-001
Policy MGT-POL-001: Faculty Development and Continuing Professional Education Policy. Department: Management Department (Operational Academic Unit). Category: Faculty Affairs. Defines the institution's commitment to faculty skill enhancement through training, workshops, conferences, sabbaticals, and ...

Chunk 3
Policy: QAA-POL-003
Policy QAA-POL-003: Student Learning Outcomes Assessment and Program

In [36]:
whole_policy_results = pd.DataFrame(results)

whole_policy_results.head()

,Embedding,Question,Expected,Retrieved,Correct
0,MiniLM,Who approves proposed curriculum changes?,BAD-POL-001,BAD-POL-001,True
1,MiniLM,How often must academic programs undergo inter...,BAD-POL-001,BAD-POL-001,True
2,MiniLM,When is the Academic Program Review Policy app...,BAD-POL-001,BAD-POL-001,True
3,MiniLM,Who updates the program catalog after approval?,BAD-POL-001,BAD-POL-001,True
4,MiniLM,What happens if a proposal is deferred?,BAD-POL-001,FIN-POL-001,False


In [37]:
summary = (
    whole_policy_results
    .groupby("Embedding")["Correct"]
    .agg(["sum", "count"])
)

summary["Accuracy (%)"] = (
    summary["sum"] / summary["count"] * 100
).round(2)

summary

,sum,count,Accuracy (%)
Embedding,,,
BGE,38,44,86.36
MiniLM,38,44,86.36
Nomic,36,44,81.82


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def fixed_chunking(data,
                   chunk_size=500,
                   chunk_overlap=100):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    documents = []

    for department in data["departments"]:
        for policy in department["policies"]:

            chunks = splitter.split_text(policy["rag_chunk"])

            for i, chunk in enumerate(chunks):

                documents.append({
                    "id": f'{policy["policy_id"]}_{i}',
                    "text": chunk,
                    "metadata": {
                        "policy_id": policy["policy_id"],
                        "department": policy["department_name"],
                        "category": policy["category"],
                        "title": policy["title"]
                    }
                })

    return documents


documents = fixed_chunking(data)

print(len(documents))
print(documents[:2])

89
[{'id': 'BAD-POL-001_0', 'text': 'Policy BAD-POL-001: Academic Program Review and Approval Policy. Department: Business Administration Department (Operational Academic Unit). Category: Curriculum Governance. This policy defines the process for initiating, reviewing, modifying, or discontinuing academic programs. Programs must undergo internal review every five years and must align with national higher education standards and affiliating university regulations. Scope: All undergraduate and postgraduate programs. Key roles:', 'metadata': {'policy_id': 'BAD-POL-001', 'department': 'Business Administration Department', 'category': 'Curriculum Governance', 'title': 'Academic Program Review and Approval Policy'}}, {'id': 'BAD-POL-001_1', 'text': 'affiliating university regulations. Scope: All undergraduate and postgraduate programs. Key roles: Department Head initiates review; Review Committee evaluates curriculum; Academic Council approves changes; Registrar updates catalog. Triggered wh

In [23]:
fixed_retrievers = benchmark_embeddings(
    documents,
    "FixedChunk"
)

Chunking : FixedChunk
Embedding: MiniLM
Done.

Chunking : FixedChunk
Embedding: BGE
Done.

Chunking : FixedChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [24]:
# question = "Who approves curriculum changes?"

# for model_name, retriever in fixed_retrievers.items():

#     print(model_name)

#     docs = retriever.invoke(question)

#     print(docs[0].page_content)

In [25]:
import pandas as pd

results = []

for model_name, retriever in fixed_retrievers.items():

    for sample in ground_truth:

        question = sample["question"]
        expected = sample["policy_id"]

        docs = retriever.invoke(question)

        retrieved = docs[0].metadata.get("policy_id", "Unknown") if docs else None

        results.append({
            "Embedding": model_name,
            "Question": question,
            "Expected": expected,
            "Retrieved": retrieved,
            "Correct": expected == retrieved
        })

fixed_results = pd.DataFrame(results)

summary = (
    fixed_results
    .groupby("Embedding")["Correct"]
    .agg(["sum", "count"])
)

summary["Accuracy (%)"] = (
    summary["sum"] / summary["count"] * 100
).round(2)

print(summary)

fixed_results.head()

           sum  count  Accuracy (%)
Embedding                          
BGE         38     44         86.36
MiniLM      37     44         84.09
Nomic       40     44         90.91


,Embedding,Question,Expected,Retrieved,Correct
0,MiniLM,Who approves proposed curriculum changes?,BAD-POL-001,BAD-POL-001,True
1,MiniLM,How often must academic programs undergo inter...,BAD-POL-001,BAD-POL-001,True
2,MiniLM,When is the Academic Program Review Policy app...,BAD-POL-001,BAD-POL-001,True
3,MiniLM,Who updates the program catalog after approval?,BAD-POL-001,BAD-POL-001,True
4,MiniLM,What happens if a proposal is deferred?,BAD-POL-001,FIN-POL-001,False


In [26]:
def section_chunking(data):

    documents = []

    for department in data["departments"]:
        for policy in department["policies"]:

            sections = {
                "Description": policy.get("description"),
                "Scope": policy.get("scope"),
                "Decision Context": policy.get("decision_context"),
                "Legal Basis": "\n".join(policy.get("legal_regulatory_basis", [])),
                "Constraints": "\n".join(policy.get("constraints", [])),
                "Related Entities": "\n".join(policy.get("related_entities", []))
            }

            # Responsibilities
            if "key_responsibilities" in policy:
                responsibilities = "\n".join(
                    [
                        f"{role}: {task}"
                        for role, task in policy["key_responsibilities"].items()
                    ]
                )

                sections["Responsibilities"] = responsibilities

            # Outcomes
            if "outcomes" in policy:
                outcomes = "\n".join(
                    [
                        f"{k}: {v}"
                        for k, v in policy["outcomes"].items()
                    ]
                )

                sections["Outcomes"] = outcomes

            for section_name, text in sections.items():

                if text:

                    documents.append({

                        "id": f'{policy["policy_id"]}_{section_name}',

                        "text": text,

                        "metadata": {

                            "policy_id": policy["policy_id"],
                            "section": section_name,
                            "department": policy["department_name"],
                            "category": policy["category"],
                            "title": policy["title"]
                        }
                    })

    return documents


documents = section_chunking(data)

print(len(documents))
print(documents[:5])

240
[{'id': 'BAD-POL-001_Description', 'text': 'Defines the process for initiating, reviewing, modifying, or discontinuing academic programs. Programs must undergo internal review every five years and must align with national higher education standards and affiliating university regulations.', 'metadata': {'policy_id': 'BAD-POL-001', 'section': 'Description', 'department': 'Business Administration Department', 'category': 'Curriculum Governance', 'title': 'Academic Program Review and Approval Policy'}}, {'id': 'BAD-POL-001_Scope', 'text': 'All undergraduate and postgraduate programs offered by the department.', 'metadata': {'policy_id': 'BAD-POL-001', 'section': 'Scope', 'department': 'Business Administration Department', 'category': 'Curriculum Governance', 'title': 'Academic Program Review and Approval Policy'}}, {'id': 'BAD-POL-001_Decision Context', 'text': 'Used when: proposing a new program, revising an existing curriculum, responding to accreditation feedback, or phasing out an 

In [27]:
section_retreivers = benchmark_embeddings(
    documents,
    "SectionChunk"
)

Chunking : SectionChunk
Embedding: MiniLM
Done.

Chunking : SectionChunk
Embedding: BGE
Done.

Chunking : SectionChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [28]:
# question = "Who approves curriculum changes?"

# for model_name, retriever in section_retreivers.items():

#     print(model_name)

#     docs = retriever.invoke(question)

#     print(docs[0].page_content)

In [29]:
import pandas as pd

results = []

for model_name, retriever in section_retreivers.items():

    for sample in ground_truth:

        question = sample["question"]
        expected = sample["policy_id"]

        docs = retriever.invoke(question)

        retrieved = docs[0].metadata.get("policy_id", "Unknown") if docs else None

        results.append({
            "Embedding": model_name,
            "Question": question,
            "Expected": expected,
            "Retrieved": retrieved,
            "Correct": expected == retrieved
        })

section_results = pd.DataFrame(results)

summary = (
    section_results
    .groupby("Embedding")["Correct"]
    .agg(["sum", "count"])
)

summary["Accuracy (%)"] = (
    summary["sum"] / summary["count"] * 100
).round(2)

print(summary)

section_results.head()

           sum  count  Accuracy (%)
Embedding                          
BGE         42     44         95.45
MiniLM      40     44         90.91
Nomic       41     44         93.18


,Embedding,Question,Expected,Retrieved,Correct
0,MiniLM,Who approves proposed curriculum changes?,BAD-POL-001,BAD-POL-001,True
1,MiniLM,How often must academic programs undergo inter...,BAD-POL-001,BAD-POL-001,True
2,MiniLM,When is the Academic Program Review Policy app...,BAD-POL-001,BAD-POL-001,True
3,MiniLM,Who updates the program catalog after approval?,BAD-POL-001,BAD-POL-001,True
4,MiniLM,What happens if a proposal is deferred?,BAD-POL-001,FIN-POL-001,False


In [30]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

splitter = SemanticChunker(embedding)


def semantic_chunking(data):

    documents = []

    for department in data["departments"]:
        for policy in department["policies"]:

            chunks = splitter.split_text(policy["rag_chunk"])

            for i, chunk in enumerate(chunks):

                documents.append({

                    "id": f'{policy["policy_id"]}_{i}',

                    "text": chunk,

                    "metadata": {

                        "policy_id": policy["policy_id"],
                        "department": policy["department_name"],
                        "category": policy["category"],
                        "title": policy["title"]
                    }
                })

    return documents


documents = semantic_chunking(data)

print(len(documents))
print(documents[:5])

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_31412\2038789377.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


60
[{'id': 'BAD-POL-001_0', 'text': 'Policy BAD-POL-001: Academic Program Review and Approval Policy. Department: Business Administration Department (Operational Academic Unit). Category: Curriculum Governance.', 'metadata': {'policy_id': 'BAD-POL-001', 'department': 'Business Administration Department', 'category': 'Curriculum Governance', 'title': 'Academic Program Review and Approval Policy'}}, {'id': 'BAD-POL-001_1', 'text': 'This policy defines the process for initiating, reviewing, modifying, or discontinuing academic programs. Programs must undergo internal review every five years and must align with national higher education standards and affiliating university regulations. Scope: All undergraduate and postgraduate programs. Key roles: Department Head initiates review; Review Committee evaluates curriculum; Academic Council approves changes; Registrar updates catalog. Triggered when proposing a new program, revising curriculum, responding to accreditation feedback, or phasing o

In [31]:
semantic_retrievers = benchmark_embeddings(
    documents,
    "SemanticChunk"
)

Chunking : SemanticChunk
Embedding: MiniLM
Done.

Chunking : SemanticChunk
Embedding: BGE
Done.

Chunking : SemanticChunk
Embedding: Nomic


<All keys matched successfully>


Done.



In [32]:
# question = "Who approves curriculum changes?"

# for model_name, retriever in semantic_retrievers.items():

#     print(model_name)

#     docs = retriever.invoke(question)

#     print(docs[0].page_content)

In [33]:
import pandas as pd

results = []

for model_name, retriever in semantic_retrievers.items():

    for sample in ground_truth:

        question = sample["question"]
        expected = sample["policy_id"]

        docs = retriever.invoke(question)

        retrieved = docs[0].metadata.get("policy_id", "Unknown") if docs else None

        results.append({
            "Embedding": model_name,
            "Question": question,
            "Expected": expected,
            "Retrieved": retrieved,
            "Correct": expected == retrieved
        })

results_df = pd.DataFrame(results)

results_df.head()

,Embedding,Question,Expected,Retrieved,Correct
0,MiniLM,Who approves proposed curriculum changes?,BAD-POL-001,BAD-POL-001,True
1,MiniLM,How often must academic programs undergo inter...,BAD-POL-001,BAD-POL-001,True
2,MiniLM,When is the Academic Program Review Policy app...,BAD-POL-001,BAD-POL-001,True
3,MiniLM,Who updates the program catalog after approval?,BAD-POL-001,BAD-POL-001,True
4,MiniLM,What happens if a proposal is deferred?,BAD-POL-001,FIN-POL-001,False


In [34]:
summary = (
    results_df
    .groupby("Embedding")["Correct"]
    .agg(["sum", "count"])
)

summary["Accuracy (%)"] = (
    summary["sum"] / summary["count"] * 100
).round(2)

summary

,sum,count,Accuracy (%)
Embedding,,,
BGE,38,44,86.36
MiniLM,38,44,86.36
Nomic,36,44,81.82
